# ML.ENERGY V3 文本 LLM inference 数据审计

**结论：**公开稳定榜单快照适合做 V3 已观测域内的配置级 GPU 稳态能耗强度/聚合功率原型，但不足以支持可泛化的事前单请求或整任务总能耗建模。本文只审计三个文本 LLM inference 紧凑 JSON，不读取 raw timeline，也不训练模型。

## 方法

调用仓库中的标准库审计函数，检查结构、缺失、重复、覆盖、数值范围、代数恒等式、ITL 分位顺序及官方 batch 稳定阈值的公开部分。

In [1]:
import json
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from scripts.audit_mlenergy_v3_llm import build_audit
audit = build_audit(repo_root / 'data' / 'compact' / 'llm')
print(f"records={audit['profile']['row_count']}, columns={audit['profile']['column_count']}")

records=565, columns=23


In [2]:
profile = audit['profile']
coverage = audit['coverage_analysis']
headline = {
    'tasks': profile['coverage']['task'],
    'gpu_models': profile['coverage']['gpu_model'],
    'num_gpus': profile['coverage']['num_gpus'],
    'unique_models': coverage['unique_models'],
    'nonzero_null_counts': {k: v for k, v in profile['nulls'].items() if v['count']},
    'exact_duplicates': profile['exact_duplicates'],
    'candidate_key': profile['candidate_key'],
}
print(json.dumps(headline, ensure_ascii=False, indent=2))

{
  "tasks": {
    "gpqa": 187,
    "lm-arena-chat": 328,
    "sourcegraph-fim": 50
  },
  "gpu_models": {
    "B200": 291,
    "H100": 274
  },
  "num_gpus": {
    "1": 250,
    "2": 104,
    "4": 83,
    "8": 128
  },
  "unique_models": 27,
  "nonzero_null_counts": {},
  "exact_duplicates": {
    "duplicate_groups": 0,
    "rows_in_duplicate_groups": 0
  },
  "candidate_key": {
    "fields": [
      "task",
      "model_id",
      "gpu_model",
      "num_gpus",
      "max_num_seqs",
      "tensor_parallel",
      "expert_parallel",
      "data_parallel"
    ],
    "unique_keys": 562,
    "duplicate_groups": 3,
    "rows_in_duplicate_groups": 6,
    "multiplicity": {
      "1": 559,
      "2": 3
    },
    "duplicate_group_examples": {
      "gpqa | openai/gpt-oss-20b | H100 | 1 | 96 | 1 | 1 | 1": 2,
      "lm-arena-chat | Qwen/Qwen3-30B-A3B-Instruct-2507 | H100 | 2 | 256 | 1 | 2 | 1": 2,
      "lm-arena-chat | Qwen/Qwen3-30B-A3B-Instruct-2507 | H100 | 2 | 512 | 1 | 2 | 1": 2
    }
  

In [3]:
print(json.dumps({
    'task_by_gpu': coverage['cross_tabs']['task_by_gpu'],
    'task_by_precision': coverage['cross_tabs']['task_by_precision'],
    'models_per_task': coverage['models_per_task'],
    'gpu_comparable_keys': coverage['gpu_comparable_configuration_keys'],
}, ensure_ascii=False, indent=2))

{
  "task_by_gpu": {
    "gpqa": {
      "B200": 106,
      "H100": 81
    },
    "lm-arena-chat": {
      "B200": 151,
      "H100": 177
    },
    "sourcegraph-fim": {
      "B200": 34,
      "H100": 16
    }
  },
  "task_by_precision": {
    "gpqa": {
      "bfloat16": 89,
      "fp8": 33,
      "mxfp4": 65
    },
    "lm-arena-chat": {
      "bfloat16": 270,
      "fp8": 58
    },
    "sourcegraph-fim": {
      "bfloat16": 34,
      "fp8": 16
    }
  },
  "models_per_task": {
    "gpqa": 12,
    "lm-arena-chat": 18,
    "sourcegraph-fim": 3
  },
  "gpu_comparable_keys": {
    "paired_h100_b200_keys": 108,
    "all_configuration_keys_ignoring_gpu": 454
  }
}


In [4]:
checks = audit['consistency']
print(json.dumps(checks, ensure_ascii=False, indent=2))

{
  "power_identity": {
    "formula": "avg_power_watts = energy_per_token_joules * output_throughput_tokens_per_sec",
    "checked": 565,
    "failed": 0,
    "failed_rows": [],
    "max_relative_error": 2.483118554175566e-16,
    "median_relative_error": 0.0,
    "relative_tolerance": 1e-09
  },
  "request_energy_identity": {
    "formula": "energy_per_request_joules = energy_per_token_joules * avg_output_len",
    "checked": 565,
    "failed": 0,
    "failed_rows": [],
    "max_relative_error": 0.0,
    "median_relative_error": 0.0,
    "relative_tolerance": 1e-09
  },
  "itl_percentile_order": {
    "formula": "median_itl_ms <= p90_itl_ms <= p95_itl_ms <= p99_itl_ms",
    "checked": 565,
    "failed": 0,
    "failed_rows": []
  },
  "batch_utilization": {
    "formula": "avg_batch_size / max_num_seqs",
    "checked": 565,
    "below_official_stability_threshold": 0,
    "failed_rows": [],
    "minimum": 0.8540116567460317,
    "median": 0.9982979302832243,
    "maximum": 1.0
  }
}


## 建模含义

可保留的最小事前特征是任务类别、模型结构/参数量/权重精度、GPU 型号与卡数、max_num_seqs 和并行配置。actual avg_output_len、avg_batch_size、吞吐、ITL 及所有能耗/功率结果均不得作为输入。model_id 只应用于分组和按模型外推验证。

In [5]:
selected_numeric = {field: audit['numeric_profile'][field] for field in [
    'total_params_billions', 'activated_params_billions',
    'avg_power_watts', 'energy_per_token_joules',
    'energy_per_request_joules', 'avg_output_len'
]}
print(json.dumps(selected_numeric, ensure_ascii=False, indent=2))

{
  "total_params_billions": {
    "count": 565,
    "minimum": 8.0,
    "q1": 14.0,
    "median": 32.0,
    "q3": 235.0,
    "maximum": 671.0,
    "non_positive": 0,
    "iqr_outlier_count": 22
  },
  "activated_params_billions": {
    "count": 565,
    "minimum": 3.0,
    "q1": 8.0,
    "median": 17.0,
    "q3": 32.0,
    "maximum": 405.0,
    "non_positive": 0,
    "iqr_outlier_count": 68
  },
  "avg_power_watts": {
    "count": 565,
    "minimum": 293.91033781272114,
    "q1": 616.5121449284105,
    "median": 1021.968645687438,
    "q3": 3098.2878789821207,
    "maximum": 7710.24963406389,
    "non_positive": 0,
    "iqr_outlier_count": 19
  },
  "energy_per_token_joules": {
    "count": 565,
    "minimum": 0.028446538525400376,
    "q1": 0.1858720223686476,
    "median": 0.4672388908079148,
    "q3": 1.3661232958863816,
    "maximum": 34.333625414199396,
    "non_positive": 0,
    "iqr_outlier_count": 81
  },
  "energy_per_request_joules": {
    "count": 565,
    "minimum": 18.472